<a href="https://colab.research.google.com/github/govind0429/BIS_LAB/blob/main/1BM24CS106_Indoor_air_Quality_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
import copy

# 1. Represent each room as an individual cell
@dataclass
class RoomCell:
    row: int
    col: int
    temp: float          # Current temperature (°C)
    humidity: float      # Relative humidity (%)
    pollutant: float     # CO2 concentration (ppm)
    occupancy: int       # Number of people in the room
    vent_level: float    # Fan speed (0.0 to 1.0)
    fitness: float = float('inf')

class ParallelCellularIAQOptimizer:
    def __init__(self, rows: int, cols: int, max_iter: int, initial_grid):
        self.rows = rows
        self.cols = cols
        self.max_iter = max_iter

        # Target comfort & safety thresholds
        self.t_opt = 22.0       # Optimal temperature (°C)
        self.p_max = 800.0      # Safe CO2 limit (ppm)

        # Fitness weights
        self.w_air = 0.5
        self.w_temp = 0.3
        self.w_energy = 0.2

        # Load user-provided grid
        self.grid = initial_grid
        self.best_global_fitness = float('inf')
        self.optimal_grid = copy.deepcopy(self.grid)

    # 3. Evaluate fitness based on air quality and energy consumption
    def _evaluate_cell_fitness(self, cell: RoomCell) -> float:
        air_penalty = max(0.0, (cell.pollutant - self.p_max) / self.p_max)
        temp_penalty = abs(cell.temp - self.t_opt) / self.t_opt
        energy_penalty = cell.vent_level ** 2

        cell.fitness = (
            self.w_air * air_penalty +
            self.w_temp * temp_penalty +
            self.w_energy * energy_penalty
        )
        return cell.fitness

    # Helper: Get adjacent rooms (Moore neighborhood)
    def _get_neighbors(self, r: int, c: int):
        neighbors = []
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0:
                    continue
                nr, nc = r + dr, c + dc
                if 0 <= nr < self.rows and 0 <= nc < self.cols:
                    neighbors.append(self.grid[nr][nc])
        return neighbors

    # 4. Update ventilation levels based on neighboring room conditions
    def _update_single_cell(self, coords):
        r, c = coords
        current = copy.deepcopy(self.grid[r][c])
        neighbors = self._get_neighbors(r, c)

        if neighbors:
            best_neighbor = min(neighbors, key=lambda n: n.fitness)
            avg_neighbor_pollutant = np.mean([n.pollutant for n in neighbors])

            learning_rate = 0.4
            diffusion_factor = 0.05 * (avg_neighbor_pollutant - current.pollutant) / self.p_max
            occupancy_boost = 0.015 * current.occupancy
            noise = np.random.normal(0, 0.01)

            new_vent = (
                current.vent_level +
                learning_rate * (best_neighbor.vent_level - current.vent_level) +
                diffusion_factor +
                occupancy_boost +
                noise
            )
            current.vent_level = float(np.clip(new_vent, 0.05, 1.0))

        # Simulate environment response for the next time step
        co2_generation = current.occupancy * 18.0
        co2_removal = current.vent_level * 250.0
        current.pollutant = max(400.0, current.pollutant + co2_generation - co2_removal)
        current.temp += (self.t_opt - current.temp) * (0.2 * current.vent_level)

        return current

    # 5. Run parallel iterations to identify an efficient configuration
    def optimize(self):
        all_coords = [(r, c) for r in range(self.rows) for c in range(self.cols)]

        with ThreadPoolExecutor() as executor:
            for iteration in range(self.max_iter):
                # Evaluate fitness across all cells in parallel
                all_cells = [self.grid[r][c] for r, c in all_coords]
                list(executor.map(self._evaluate_cell_fitness, all_cells))

                # Track best building-wide configuration
                total_fitness = sum(cell.fitness for cell in all_cells)
                if total_fitness < self.best_global_fitness:
                    self.best_global_fitness = total_fitness
                    self.optimal_grid = copy.deepcopy(self.grid)

                # Update all cells simultaneously
                updated_cells = list(executor.map(self._update_single_cell, all_coords))

                next_grid = [[None] * self.cols for _ in range(self.rows)]
                for cell in updated_cells:
                    next_grid[cell.row][cell.col] = cell
                self.grid = next_grid

        return self.optimal_grid

# 2. Take User Input to Initialize the Grid
def get_user_grid_input():
    print("=== Building Ventilation Optimizer Setup ===")
    rows = int(input("Enter number of rows (building zones): "))
    cols = int(input("Enter number of columns (building zones): "))
    max_iter = int(input("Enter number of optimization iterations (e.g., 30): "))

    mode = input("Enter room data manually? (y/n - 'n' generates random values): ").strip().lower()
    rng = np.random.default_rng()
    grid = []

    for r in range(rows):
        row_cells = []
        for c in range(cols):
            if mode == 'y':
                print(f"\n--- Enter initial details for Room ({r}, {c}) ---")
                temp = float(input("  Temperature in °C (e.g., 25.0): "))
                humidity = float(input("  Humidity in % (e.g., 55.0): "))
                pollutant = float(input("  CO2 Pollutant level in ppm (e.g., 900): "))
                occupancy = int(input("  Number of people in room (e.g., 5): "))
                vent_pct = float(input("  Initial fan speed % (0 to 100): "))
                vent_level = np.clip(vent_pct / 100.0, 0.0, 1.0)
            else:
                occupancy = int(rng.integers(0, 15))
                temp = float(rng.uniform(20.0, 28.0))
                humidity = float(rng.uniform(40.0, 65.0))
                pollutant = float(400 + occupancy * 35 + rng.uniform(0, 100))
                vent_level = float(rng.uniform(0.2, 0.8))

            row_cells.append(RoomCell(r, c, temp, humidity, pollutant, occupancy, vent_level))
        grid.append(row_cells)

    return rows, cols, max_iter, grid

# --- Main Execution ---
if __name__ == "__main__":
    rows, cols, max_iter, user_grid = get_user_grid_input()

    optimizer = ParallelCellularIAQOptimizer(rows, cols, max_iter, user_grid)
    best_config = optimizer.optimize()

    print(f"\n==========================================")
    print(f"Optimal Building Total Fitness Score: {optimizer.best_global_fitness:.4f}")
    print(f"==========================================\n")
    print("Optimized Room Ventilation Configuration:")
    for r in range(rows):
        for c in range(cols):
            room = best_config[r][c]
            print(
                f"Room ({r},{c}) | People: {room.occupancy:2d} | "
                f"CO2: {room.pollutant:6.1f} ppm | Temp: {room.temp:4.1f}°C | "
                f"Fan Speed: {room.vent_level * 100:5.1f}%"
            )


=== Building Ventilation Optimizer Setup ===
Enter number of rows (building zones): 2
Enter number of columns (building zones): 2
Enter number of optimization iterations (e.g., 30): 30
Enter room data manually? (y/n - 'n' generates random values): y

--- Enter initial details for Room (0, 0) ---
  Temperature in °C (e.g., 25.0): 22
  Humidity in % (e.g., 55.0): 45
  CO2 Pollutant level in ppm (e.g., 900): 430
  Number of people in room (e.g., 5): 1
  Initial fan speed % (0 to 100): 30

--- Enter initial details for Room (0, 1) ---
  Temperature in °C (e.g., 25.0): 26.5
  Humidity in % (e.g., 55.0): 60
  CO2 Pollutant level in ppm (e.g., 900): 1150
  Number of people in room (e.g., 5): 16
  Initial fan speed % (0 to 100): 40

--- Enter initial details for Room (1, 0) ---
  Temperature in °C (e.g., 25.0): 23.5
  Humidity in % (e.g., 55.0): 50
  CO2 Pollutant level in ppm (e.g., 900): 620
  Number of people in room (e.g., 5): 4
  Initial fan speed % (0 to 100): 35

--- Enter initial detai